# Location Extraction: Multi-Model Comparison

This notebook compares 5 different approaches for extracting death counts from conflict event descriptions:

1. **NT5-small** - Numerical reasoning specialist
2. **Flan-T5-base** - Instruction-following generalist
2. **Flan-T5-Large** - Large instruction-following generalist
4. **mT5-base** - Multilingual T5 model (100+ languages)
5. **Flan-T5-XL** - Large instruction-following generalist

**Dataset**: Armed Assault + Bombing incidents (~4,300 samples, 37% zeros, 63% with fatalities)

## Notebook Structure

1. **Setup & Configuration** - Environment setup and library imports
2. **Data Management** - Load, filter, and split data
3. **Model Setup** - Load and configure models
4. **Model Training** - Train and evaluate each model
5. **Results & Analysis** - Compare models and analyze errors


#### About Structured Output Format

This notebook trains a T5-base model to extract location information from incident summaries using a **structured output format** with explicit labels for each administrative level.

1. **Data Source**
   - Using `location_info_augmented.csv` which contains human-annotated and augmented location data
   - All duplicates between block/other_areas and village names have been removed

2. **Structured Output Format**

   - **Format:** `"state: Chhattisgarh, district: Sukma, village: Nilamadgu, other_locations: Dornapal"` (more robust)
   - **Also tried:** `"state: Chhattisgarh, district: Sukma, block: Dornapal, village: Nilamadgu"` (block often missing)
   
   **Benefits:**
   - Clear hierarchy with labeled levels
   - `other_locations` provides flexible context (blocks, forests, police stations, etc.)

#### About Data Splits

- **Temporal Split:**
  - Training on earlier data, testing on more recent data (realistic deployment scenario)
  - Training ends before Telangana formation (June 2, 2014)
  - Test set contains only post-2014 incidents with current state names
- **Reasoning:**
  - Temporal split better simulates real-world deployment scenario
  - Keeps recent data for evaluation (important for model performance)
  - Avoids data leakage between train/val/test sets if some regions dominate a particular time period


#### Metrics

- **Training Metrics:**
  - **Cross-entropy loss** (`eval_loss`) on validation set
  - **BLEU/ROUGE metrics** not used (better suited for text generation than structured extraction)
- **Evaluation Metrics:**
  - **Exact Match Accuracy**: All levels must match perfectly
  - **Per-Level Metrics**: Precision, Recall, F1 for each of state/district/village/other_locations
  - **Micro-averaged F1**: Overall performance across all levels
  - **Level-by-level breakdown**: See exactly which administrative levels the model struggles with
- **Exact versus fuzzy matching**:
  - For each level, we compute exact match and fuzzy match metrics
  - We use `rapidfuzz` and an 80% threshold for fuzzy matching


## 1. Setup and Configuration

### 1.1 Environment Setup

#### 1.1.1 Colab Setup

In [ ]:
# 1) Mount Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# 2) Clone or update the repo
BRANCH = "main"  # Change if working on a different branch

# Clone fresh each session 
!rm -rf /content/code-satp
!git clone -b $BRANCH --depth 1 https://github.com/eteitelbaum/code-satp.git /content/code-satp

# 3) Install required packages
!pip install -qU pip setuptools wheel
!pip install --upgrade transformers datasets evaluate rouge-score rapidfuzz accelerate bitsandbytes peft

# 4) Make result directories
import pathlib
import sys
pathlib.Path("/content/drive/MyDrive/colab/satp-results").mkdir(parents=True, exist_ok=True)
pathlib.Path("/content/drive/MyDrive/colab/satp-results/location-extraction").mkdir(parents=True, exist_ok=True)

# 5) Import utilities from cloned repo (using importlib to handle hyphens in path)
try:
    import importlib.util
    
    # Import file_io utilities
    spec = importlib.util.spec_from_file_location(
        "file_io",
        "/content/code-satp/models/classification-models/utils/file_io.py"
    )
    file_io = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(file_io)
    
    get_task_results_dir = file_io.get_task_results_dir
    save_dataframe_csv = file_io.save_dataframe_csv
    
    # Add location-models to path FIRST to prioritize local utilities
    sys.path.insert(0, "/content/code-satp/models/location-models")
    
    # Import training utilities from count-models (if needed)
    sys.path.append("/content/code-satp/models/count-models")
    
    # Define task name for consistent file organization
    TASK_NAME = "location-extraction"
    
    print("="*80)
    print("✅ SETUP COMPLETE")
    print("="*80)
    print(f"📂 Cloned repo: /content/code-satp")
    print(f"📂 Results dir: {get_task_results_dir(TASK_NAME)}")
    print(f"✅ File I/O utilities loaded successfully")
    print("="*80)
    
except Exception as e:
    print("="*80)
    print("⚠️  WARNING: Could not load file_io utilities")
    print(f"Error: {e}")
    print("="*80)
    print("Falling back to local mode - files will be saved to current directory")
    
    # Define fallback functions
    TASK_NAME = "location-extraction"
    
    def get_task_results_dir(task_name):
        return pathlib.Path(f"./results/{task_name}")
    
    def save_dataframe_csv(df, filename, task_name=None):
        output_path = f"./results/{task_name}/{filename}" if task_name else f"./{filename}"
        pathlib.Path(output_path).parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(output_path, index=False)
        return output_path
    
    print("="*80)

#### 1.1.2 Local Setup

In [ ]:
# # For local development, uncomment these lines:
# import sys
# import os

# # Local data and results paths
# DATA_PATH = "../../data/"
# RESULTS_PATH = "./results/"

# # Create local results directory
# os.makedirs(RESULTS_PATH, exist_ok=True)

# # Verify GPU availability
# import torch
# if torch.cuda.is_available():
#     print(f"✅ GPU Available: {torch.cuda.get_device_name(0)}")
# else:
#     print("⚠️ GPU not available, using CPU")

# print("✅ Local setup complete!")

### 1.2 Import Libraries

In [ ]:
# Core libraries
import os
import re
import json
import pandas as pd
import numpy as np
from pathlib import Path

# Machine learning and transformers
import torch
from torch.utils.data import Dataset
from transformers import (
    T5Tokenizer, 
    T5ForConditionalGeneration,
    Trainer, 
    TrainingArguments,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq
)
from datasets import Dataset as HFDataset

# Evaluation metrics
import evaluate
from sklearn.model_selection import train_test_split

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

## 2. Data Loading and Preparation

### 2.1 Load Raw Data

In [ ]:
# Try loading from cloned repository first
try:
    data = pd.read_csv('/content/code-satp/data/location_info_augmented.csv', dtype=str)
    print("✅ Data loaded successfully from cloned repository")
    print(f"Rows: {len(data)}, Columns: {len(data.columns)}")
except Exception as e:
    print(f"❌ Error loading CSV from cloned repo: {e}")
    print("🔧 Fallback: Trying GitHub URL...")
    
    # Fallback to GitHub if cloned repo fails or if working locally
    url = "https://raw.githubusercontent.com/eteitelbaum/code-satp/refs/heads/main/data/location_info_augmented.csv"
    try:
        data = pd.read_csv(url, dtype=str)
        print("✅ Data loaded successfully from GitHub")
        print(f"Rows: {len(data)}, Columns: {len(data.columns)}")
    except Exception as e:
        print(f"❌ Error loading CSV from GitHub: {e}")

### 2.2 Create Structured Location Labels

#### Build Structured Location Column

In [ ]:
from utils.data_utils import build_structured_location

# Create structured human_annotated_location column from individual location fields
# Apply to create the human_annotated_location column
data['human_annotated_location'] = data.apply(build_structured_location, axis=1)

# Display sample to verify
print("Sample human-annotated locations:")
display(data[['state', 'district', 'village_name', 'other_locations', 'human_annotated_location']].head(10))

#### Verify Data Structure

In [ ]:
# Display the first few rows to verify the data structure
print("Dataset shape:", data.shape)
print("\nFirst few rows:")
display(data.head())

### 2.3 Data Preprocessing and Filtering

#### Create Train/Validation/Test Splits

In [ ]:
# TEMPORAL SPLIT: Sort by date for chronological train/val/test split

# Training on earlier data, testing on more recent data (realistic deployment scenario)
data['date'] = pd.to_datetime(data['date'], errors='coerce')
data = data.sort_values('date').reset_index(drop=True)

# Calculate split indices for 80/10/10 split
train_end_idx = int(len(data) * 0.8)
val_end_idx = int(len(data) * 0.9)

# Create temporal splits
train_data = data.iloc[:train_end_idx]
val_data = data.iloc[train_end_idx:val_end_idx]
test_data = data.iloc[val_end_idx:]

print(f"Temporal split summary:")
print(f"  Training:   {len(train_data)} incidents ({train_data['date'].min()} to {train_data['date'].max()})")
print(f"  Validation: {len(val_data)} incidents ({val_data['date'].min()} to {val_data['date'].max()})")
print(f"  Test:       {len(test_data)} incidents ({test_data['date'].min()} to {test_data['date'].max()})")
print(f"\nNote: Training ends before Telangana formation (June 2, 2014)")
print(f"      Test set contains only post-2014 incidents with current state names")




### 2.4 Save Data Splits

In [ ]:
# Save splits to disk using file_io utilities

def save_location_data_splits(train_df, val_df, test_df, task_name=TASK_NAME):
    """Store prepared location extraction splits using standard file_io helpers."""
    priority_columns = [
        'incident_number',
        'incident_summary',
        'date',
        'state',
        'district',
        'village_name',
        'other_locations',
        'human_annotated_location'
    ]

    def reorder_columns(df):
        ordered_cols = [col for col in priority_columns if col in df.columns]
        return df[ordered_cols].copy()

    artifacts = {
        'train.csv': reorder_columns(train_df),
        'val.csv': reorder_columns(val_df),
        'test.csv': reorder_columns(test_df)
    }

    saved_paths = []
    for filename, prepared_df in artifacts.items():
        saved_paths.append(save_dataframe_csv(prepared_df, filename, task_name=task_name))

    print("✅ Saved data splits:")
    for path in saved_paths:
        print(f"  📁 {path}")


save_location_data_splits(train_data, val_data, test_data)


### 2.5 Display Sample Data

In [ ]:
# Display sample data from each split

from utils.data_utils import preview_location_examples

preview_location_examples(train_data, "Train Split Examples")
preview_location_examples(val_data, "Validation Split Examples")
preview_location_examples(test_data, "Test Split Examples")

## 3. Model Setup

**Training Strategy:**
- **10 epochs maximum** with **early stopping** (patience=2 epochs)
- Training stops if validation loss doesn't improve for 2 consecutive epochs
- Prevents overfitting while allowing model to fully converge
- Final model is the one with lowest validation loss

### 3.1 Training Configuration

In [ ]:
# Storage for results
all_results = {}
all_predictions = {}

# Training configuration constants
BATCH_SIZE = 8
MAX_EPOCHS = 10
LEARNING_RATE_SEQ2SEQ = 3e-5
LEARNING_RATE_ENCODER = 2e-5
GENERATION_MAX_LENGTH = 64 
SEED = 42

print("Configuration:")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Max epochs: {MAX_EPOCHS} (early stopping enabled with patience=2)")
print(f"  LR (seq2seq): {LEARNING_RATE_SEQ2SEQ}")
print(f"  LR (encoder): {LEARNING_RATE_ENCODER}")
print(f"  Generation max length: {GENERATION_MAX_LENGTH}")

### 3.2 Prepare Data Dictionaries

In [ ]:
# Prepare input/target dicts from temporal splits (shared across models)
from utils.data_utils import prepare_location_seq2seq_data, make_tokenized_seq2seq_datasets

print("Preparing data dictionaries for location extraction...")
train_dict = prepare_location_seq2seq_data(train_data)
val_dict = prepare_location_seq2seq_data(val_data)
test_dict = prepare_location_seq2seq_data(test_data)

print(f"\nData prepared:")
print(f"  Training samples: {len(train_dict['input'])}")
print(f"  Validation samples: {len(val_dict['input'])}")
print(f"  Test samples: {len(test_dict['input'])}")

# Example
print("\nExample input:")
print(train_dict['input'][0][:200] + "...")
print(f"\nExample target:")
print(train_dict['target'][0])

## 4. Model Training and Evaluation

Train and evaluate seq2seq models for location extraction:
1. **Flan-T5-base** - Instruction-following generalist (250M parameters)
2. **Flan-T5-large** - Larger instruction-following model (780M parameters)
3. **IndicBART** - India-specific multilingual model
4. **mT5-base** - Multilingual T5 (100+ languages)
5. **Flan-T5-XL (QLoRA)** - Largest model with LoRA fine-tuning (3B parameters)

Each model uses the shared data prepared in Section 3 and the `run_seq2seq_location_model` utility for training/evaluation.

Note: Each model tokenizes its own inputs with its own tokenizer; all models share the same prepared text dictionaries (train_dict, val_dict, test_dict).

### 4.1 Model 1: Flan-T5-base

Train Flan-T5-base using the shared runner utility

In [ ]:
from utils.data_utils import make_tokenized_seq2seq_datasets
from utils.model_utils import run_seq2seq_location_model

# Tokenize with the model's own tokenizer
train_dataset, val_dataset, test_dataset = make_tokenized_seq2seq_datasets(
    "google/flan-t5-base", train_dict, val_dict, test_dict,
    max_input_length=512, max_target_length=128
)

flan_t5_base_metrics = run_seq2seq_location_model(
    model_id='google/flan-t5-base',
    name='flan-t5-base',
    train_dataset=train_dataset,  
    val_dataset=val_dataset,      
    test_dataset=test_dataset,    
    test_df=test_data,           
    task_name=TASK_NAME,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE_SEQ2SEQ,
    num_epochs=MAX_EPOCHS,
    generation_max_length=GENERATION_MAX_LENGTH,
    seed=SEED,
    generation_num_beams=4
)

# Store results
all_results['flan-t5-base'] = flan_t5_base_metrics

### 4.2 Model 2: Flan-T5-large
Train Flan-T5-large using the shared runner utility

In [ ]:
from utils.data_utils import make_tokenized_seq2seq_datasets

# Tokenize with the model's own tokenizer
train_dataset, val_dataset, test_dataset = make_tokenized_seq2seq_datasets(
    "google/flan-t5-large", train_dict, val_dict, test_dict,
    max_input_length=512, max_target_length=128
)

flan_t5_large_metrics = run_seq2seq_location_model(
    model_id='google/flan-t5-large',
    name='flan-t5-large',
    train_dataset=train_dataset, 
    val_dataset=val_dataset,     
    test_dataset=test_dataset,    
    test_df=test_data,            
    task_name=TASK_NAME,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE_SEQ2SEQ,
    num_epochs=MAX_EPOCHS,
    generation_max_length=GENERATION_MAX_LENGTH,
    seed=SEED,
    generation_num_beams=4
)

# Store results
all_results['flan-t5-large'] = flan_t5_large_metrics

### 4.3 Model 3: IndicBART
Train IndicBART using the shared runner utility

In [ ]:
from utils.data_utils import make_tokenized_seq2seq_datasets

# Tokenize with the model's own tokenizer
train_dataset, val_dataset, test_dataset = make_tokenized_seq2seq_datasets(
    "ai4bharat/IndicBARTSS", train_dict, val_dict, test_dict,
    max_input_length=512, max_target_length=128
)

indicbart_metrics = run_seq2seq_location_model(
    model_id='ai4bharat/IndicBARTSS',
    name='indicbart',
    train_dataset=train_dataset,  
    val_dataset=val_dataset,      
    test_dataset=test_dataset,    
    test_df=test_data,           
    task_name=TASK_NAME,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE_SEQ2SEQ,
    num_epochs=MAX_EPOCHS,
    generation_max_length=GENERATION_MAX_LENGTH,
    seed=SEED,
    generation_num_beams=4
)

# Store results
all_results['indicbart'] = indicbart_metrics

### 4.4 Model 4: mT5-base
Train mT5-base using the shared runner utility
Note: mT5 uses the '<2en>' language tag prefix for English inputs

In [ ]:
from utils.data_utils import make_tokenized_seq2seq_datasets

# Tokenize with the model's own tokenizer
train_dataset, val_dataset, test_dataset = make_tokenized_seq2seq_datasets(
    "google/mt5-base", train_dict, val_dict, test_dict,
    max_input_length=512, max_target_length=128
)

mt5_metrics = run_seq2seq_location_model(
    model_id='google/mt5-base',
    name='mt5-base',
    train_dataset=train_dataset, 
    val_dataset=val_dataset,      
    test_dataset=test_dataset,   
    test_df=test_data,          
    task_name=TASK_NAME,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE_SEQ2SEQ,
    num_epochs=MAX_EPOCHS,
    generation_max_length=GENERATION_MAX_LENGTH,
    seed=SEED,
    generation_num_beams=4
)

# Store results
all_results['mt5-base'] = mt5_metrics

### 4.5 Model 5: Flan-T5-XL (QLoRA)

Train Flan-T5-XL with QLoRA using the dedicated LoRA runner

This model requires 4-bit quantization and LoRA adapters due to its size (3B parameters)

**Requires:** pip install peft bitsandbytes accelerate

In [ ]:
from utils.data_utils import make_tokenized_seq2seq_datasets
from utils.model_utils import run_flan_t5_xl_lora_location_model

# Tokenize with the model's own tokenizer
train_dataset, val_dataset, test_dataset = make_tokenized_seq2seq_datasets(
    "google/flan-t5-xl", train_dict, val_dict, test_dict,
    max_input_length=512, max_target_length=128
)

flan_t5_xl_metrics = run_flan_t5_xl_lora_location_model(
    train_dataset=train_dataset,  
    val_dataset=val_dataset,      
    test_dataset=test_dataset,    
    test_df=test_data,           
    task_name=TASK_NAME,
    batch_size=2,  # Smaller batch size for memory efficiency
    learning_rate=5e-5,
    num_epochs=3,  # Fewer epochs typically needed with LoRA
    generation_max_length=GENERATION_MAX_LENGTH,
    seed=SEED,
    generation_num_beams=4
)

# Store results
all_results['flan-t5-xl-lora'] = flan_t5_xl_metrics

## 5. Results Comparison

Compare all models across multiple metrics by loading previously saved results.

**Workflow:**
- Load metrics and predictions from Section 4 model training runs
- Create comparison tables for overall and per-level metrics
- Visualize model performance
- Combine all predictions into a single file

**Note:** Models are trained and evaluated in Section 4. This section only loads and compares the saved results.

### 5.1 Load Previously Saved Results

If any models were run previously and saved, load them here. This allows resuming after failures.

In [ ]:
# Load any previously saved results
print("Loading previously saved results...")
print("=" * 80)

results_dir = get_task_results_dir(TASK_NAME)
model_names = ['flan-t5-base', 'flan-t5-large', 'indicbart', 'mt5-base', 'flan-t5-xl-lora']

loaded_count = 0
skipped_count = 0

for model_name in model_names:
    # Check if this model already has in-memory results
    if model_name in all_results and all_results[model_name] is not None:
        print(f"✓ {model_name}: Already in memory (from this session)")
        skipped_count += 1
        continue
    
    # Try to load from disk
    metrics_path = results_dir / f"location_{model_name}_metrics.json"
    predictions_path = results_dir / f"location_{model_name}_predictions.csv"
    
    if metrics_path.exists() and predictions_path.exists():
        try:
            # Load metrics
            with open(metrics_path, 'r') as f:
                metrics = json.load(f)
            
            # Load predictions
            preds_df = pd.read_csv(predictions_path)
            
            # Store in dictionaries
            all_results[model_name] = metrics
            prediction_dict = {
                'predictions': preds_df['predicted_location'].values if 'predicted_location' in preds_df.columns else None,
                'labels': preds_df['true_location'].values if 'true_location' in preds_df.columns else None
            }
            # Preserve identifiers and context if present
            if 'incident_number' in preds_df.columns:
                prediction_dict['incident_number'] = preds_df['incident_number'].values
            if 'incident_summary' in preds_df.columns:
                prediction_dict['incident_summary'] = preds_df['incident_summary'].values

            all_predictions[model_name] = prediction_dict
            
            print(f"✓ {model_name}: Loaded from disk")
            loaded_count += 1
        except Exception as e:
            print(f"✗ {model_name}: Error loading - {e}")
    else:
        print(f"✗ {model_name}: No saved results found")

print("=" * 80)
print(f"Summary: {skipped_count} in memory, {loaded_count} loaded from disk, {len(model_names) - skipped_count - loaded_count} missing")
print(f"Total available: {len([m for m in model_names if m in all_results and all_results[m] is not None])}/{len(model_names)} models")
print("=" * 80)

### 5.2 Create Comparison DataFrame

In [ ]:
print("\n" + "="*80)
print("RESULTS COMPARISON")
print("="*80)

# Create comparison dataframes
overall_comparison_data = []
level_comparison_data = []
available_models = []
missing_models = []

for model_name in ['flan-t5-base', 'flan-t5-large', 'indicbart', 'mt5-base', 'flan-t5-xl-lora']:
    if model_name in all_results and all_results[model_name] is not None:
        metrics = all_results[model_name]
        
        # Overall metrics row
        overall_row = {'Model': model_name}
        if 'overall' in metrics:
            overall_row.update(metrics['overall'])
        overall_comparison_data.append(overall_row)
        
        # Per-level metrics rows
        if 'levels' in metrics:
            for level_name, level_metrics in metrics['levels'].items():
                level_row = {
                    'Model': model_name,
                    'Level': level_name
                }
                level_row.update(level_metrics)
                level_comparison_data.append(level_row)
        
        available_models.append(model_name)
    else:
        missing_models.append(model_name)

print(f"\nComparing {len(available_models)} models: {', '.join(available_models)}")
if missing_models:
    print(f"⚠️  Missing results for: {', '.join(missing_models)}")
print()

if len(overall_comparison_data) == 0:
    print("❌ No model results available for comparison!")
    print("   Please run at least one model training section first.")
else:
    # Overall metrics dataframe
    overall_df = pd.DataFrame(overall_comparison_data)
    
    # Display overall results table
    print("\n### Overall Model Comparison ###\n")
    display_cols = ['Model', 'exact_match', 'exact_core_match', 'fuzzy_match', 'fuzzy_core_match', 
                    'micro_exact_f1', 'micro_fuzzy_f1', 'total_examples']
    # Only show columns that exist
    display_cols = [col for col in display_cols if col in overall_df.columns]
    print(overall_df[display_cols].to_string(index=False))
    
    # Save overall metrics
    save_dataframe_csv(overall_df, 'location_metrics_combined_overall.csv', task_name=TASK_NAME)
    print(f"\n✅ Overall metrics saved to {get_task_results_dir(TASK_NAME)}/location_metrics_combined_overall.csv")
    
    # Per-level metrics dataframe
    if len(level_comparison_data) > 0:
        levels_df = pd.DataFrame(level_comparison_data)
        
        # Display per-level table
        print("\n### Per-Level Metrics ###\n")
        level_display_cols = ['Model', 'Level', 'exact_f1', 'exact_precision', 'exact_recall', 
                              'fuzzy_f1', 'fuzzy_precision', 'fuzzy_recall', 'support']
        level_display_cols = [col for col in level_display_cols if col in levels_df.columns]
        print(levels_df[level_display_cols].to_string(index=False))
        
        # Save per-level metrics
        save_dataframe_csv(levels_df, 'location_metrics_combined_levels.csv', task_name=TASK_NAME)
        print(f"\n✅ Per-level metrics saved to {get_task_results_dir(TASK_NAME)}/location_metrics_combined_levels.csv")


### 5.3 Visualize Model Comparison

In [ ]:
# Check if overall_df exists from previous cell
if 'overall_df' not in locals() or len(overall_comparison_data) == 0:
    print("⚠️  No results available for visualization.")
    print("   Please run at least one model training section first.")
else:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # Use all available models
    valid_results = overall_df
    
    # Exact Match comparison
    if 'exact_match' in valid_results.columns:
        axes[0].bar(valid_results['Model'], valid_results['exact_match'], color='steelblue', alpha=0.7)
        axes[0].set_ylabel('Exact Match Accuracy')
        axes[0].set_title('Exact Match Performance')
        axes[0].tick_params(axis='x', rotation=45)
        axes[0].grid(axis='y', alpha=0.3)
    
    # Fuzzy Match comparison
    if 'fuzzy_match' in valid_results.columns:
        axes[1].bar(valid_results['Model'], valid_results['fuzzy_match'], color='coral', alpha=0.7)
        axes[1].set_ylabel('Fuzzy Match Accuracy')
        axes[1].set_title('Fuzzy Match Performance')
        axes[1].tick_params(axis='x', rotation=45)
        axes[1].grid(axis='y', alpha=0.3)
    
    # Micro F1 comparison
    if 'micro_exact_f1' in valid_results.columns:
        axes[2].bar(valid_results['Model'], valid_results['micro_exact_f1'], color='green', alpha=0.7)
        axes[2].set_ylabel('Micro-averaged F1')
        axes[2].set_title('Micro F1 Performance')
        axes[2].tick_params(axis='x', rotation=45)
        axes[2].grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('figures/model_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"✅ Comparison plot saved to figures/model_comparison.png")
    print(f"   Models plotted: {', '.join(valid_results['Model'].tolist())}")

### 5.4 Save All Predictions

In [ ]:
# Save all predictions (combined)
available_predictions = {model: preds for model, preds in all_predictions.items() 
                         if preds is not None}

if len(available_predictions) == 0:
    print("❌ No predictions available to save!")
    print("   Please run at least one model training section first.")
else:
    # Get true labels from any available model
    first_model = list(available_predictions.keys())[0]
    true_labels = available_predictions[first_model].get('labels')

    # Get identifiers
    incident_numbers = available_predictions[first_model].get('incident_number')
    incident_summaries = available_predictions[first_model].get('incident_summary')

    # Fallback to test_data if needed
    if incident_numbers is None and 'test_data' in locals() and 'incident_number' in test_data.columns:
        incident_numbers = test_data['incident_number'].values
    if incident_summaries is None and 'test_data' in locals() and 'incident_summary' in test_data.columns:
        incident_summaries = test_data['incident_summary'].values
    if true_labels is None and 'test_data' in locals() and 'human_annotated_location' in test_data.columns:
        true_labels = test_data['human_annotated_location'].values
    
    # Create combined dataframe with identifiers first
    combined_columns = {}
    if incident_numbers is not None:
        combined_columns['incident_number'] = incident_numbers
    if incident_summaries is not None:
        combined_columns['incident_summary'] = incident_summaries
    if true_labels is not None:
        combined_columns['true_location'] = true_labels
    
    # Add predictions from each model
    for model in available_predictions:
        preds = available_predictions[model].get('predictions')
        if preds is not None:
            combined_columns[f'{model}_pred'] = preds
    
    all_predictions_df = pd.DataFrame(combined_columns)

    # Save using file_io utilities
    save_dataframe_csv(all_predictions_df, 'location_predictions_combined.csv', task_name=TASK_NAME)
    print(f"✅ All predictions saved to {get_task_results_dir(TASK_NAME)}/location_predictions_combined.csv")
    print(f"   Predictions shape: {all_predictions_df.shape}")
    print(f"   Models included: {', '.join(available_predictions.keys())}")
    print(f"\nFirst 10 rows:")
    display(all_predictions_df.head(10))